# Week 7 exercise — QLoRA on a local GPU ("The Price Is Right", scaled up)

Week 6 fine-tuned a tiny model on CPU. Week 7 adds **4-bit quantization (QLoRA)** so a
much bigger open model fits on a modest GPU. Here: load **Qwen2.5-1.5B-Instruct** in 4-bit
on an 8GB RTX 2070, QLoRA-fine-tune it on the pricing data, and compare to the base model
and the frontier reference.

> Needs **CUDA torch + bitsandbytes** in the venv. Run with that env (note: `uv run`
> re-syncs to CPU torch — use the venv Python directly).

In [1]:
import sys, pickle, re, random
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week6/pricer/items.py").exists())
sys.path.insert(0, str(REPO / "week6"))
from pricer.items import Item  # noqa: E402
DATA = REPO / "community-contributions/NicholasDean/week6/data_small.pkl"  # reuse week 6 curation

blob = pickle.load(open(DATA, "rb"))
train = [Item.model_validate(d) for d in blob["train"]]
SAMPLE = [Item.model_validate(d) for d in blob["test"][:50]]
print("cuda:", torch.cuda.get_device_name(0), "| train:", len(train))

cuda: NVIDIA GeForce RTX 2070 SUPER | train: 16801


## Load Qwen2.5-1.5B in 4-bit (QLoRA)
4-bit NF4 quantization shrinks the weights ~4x; float16 compute (Turing GPU has no bf16).

In [2]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.pad_token or tok.eos_token
tok.truncation_side = "left"                          # keep the END (the price target)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="cuda")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
print("VRAM after load (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

W0623 11:56:50.141000 35292 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
VRAM after load (GB): 1.75


## QLoRA fine-tune (on the GPU)

In [3]:
import torch as _t

def example(it):
    # prompt ends with 'Price is $'; answer is the price -> mask labels so we train only the answer
    p = tok(it.test_prompt(), truncation=True, max_length=210)["input_ids"]
    a = tok(f"{round(it.price)}.00" + tok.eos_token, add_special_tokens=False)["input_ids"]
    return p + a, [-100] * len(p) + a

data = [example(it) for it in train[:1200]]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
model.train()
for epoch in range(3):
    random.Random(epoch).shuffle(data)
    for ids, labels in data:
        ids_t = _t.tensor([ids], device="cuda")
        out = model(input_ids=ids_t, labels=_t.tensor([labels], device="cuda"))
        out.loss.backward()
        opt.step(); opt.zero_grad()
    print(f"epoch {epoch + 1} done")
print("peak VRAM (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


c:\Users\Nicholas Dean\projects\llm_engineering\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


epoch 1 done


epoch 2 done


epoch 3 done
peak VRAM (GB): 2.49


## Compare: base Qwen vs QLoRA-fine-tuned

In [4]:
def get_price(text):
    nums = re.findall(r"[-+]?\d*\.?\d+", text.replace(",", ""))
    return float(nums[0]) if nums else 0.0

def local_price(item):
    enc = tok(item.test_prompt(), return_tensors="pt", truncation=True, max_length=220).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    return get_price(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))

def evaluate(items):
    errs = [abs(local_price(it) - it.price) for it in items]
    hits = sum(e <= 0.2 * it.price or e <= 40 for e, it in zip(errs, items))
    return sum(errs) / len(errs), hits / len(items)

model.eval()
mae_ft, hit_ft = evaluate(SAMPLE)
with model.disable_adapter():
    mae_base, hit_base = evaluate(SAMPLE)
print(f"base Qwen2.5-1.5B (4-bit):   MAE=${mae_base:,.2f}  hit-rate={hit_base:.0%}")
print(f"QLoRA fine-tuned Qwen-1.5B:   MAE=${mae_ft:,.2f}  hit-rate={hit_ft:.0%}")
print("references -> week6 distilgpt2 LoRA: $52.78 (76%) | gpt-4o-mini zero-shot: $22.80 (86%)")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


c:\Users\Nicholas Dean\projects\llm_engineering\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


base Qwen2.5-1.5B (4-bit):   MAE=$47.82  hit-rate=70%
QLoRA fine-tuned Qwen-1.5B:   MAE=$42.71  hit-rate=80%
references -> week6 distilgpt2 LoRA: $52.78 (76%) | gpt-4o-mini zero-shot: $22.80 (86%)
